# Lab 10: From Reaching to Interception
**Computational Sensorimotor Control — Fall 2026**

This lab builds the interception pipeline from the lecture: target estimation (§1–3), feedforward planning (§5), feedback control (§6), and Bayesian priors (§7).

In [ ]:
!pip3 install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(10,5), 'font.size':11, 'axes.grid':True, 'grid.alpha':0.3})
from smc import Arm, Q_REF
from smc.polar_kf import (ArcTarget, Eye, PolarTgtKF, run_diag, run_diag_MT,
                           feedforward_plan, run_trial, run_three_conditions,
                           mj_pos, mj_vel, start_hand, dt, R, HIT_WINDOW_DEG)

center = start_hand
TEAL = '#2E86AB'; ORANGE = '#E67E22'; PURPLE = '#8E44AD'
NAVY = '#1B2A4A'; RED = '#E74C3C'; GREEN = '#27AE60'; GRAY = '#7F8C8D'

def arc_pos(theta_deg):
    th = np.radians(theta_deg)
    return center + R * np.array([np.cos(th), np.sin(th)])

print('Setup complete.')

---

In [ ]:
# ── Figure 1: Interception task ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, title, mode, color in zip(axes,
    ['A: Active pursuit', 'B: Fixed gaze'],
    ['pursuit', 'fixation'], [TEAL, ORANGE]):
    theta_arc = np.linspace(0, 70, 200)
    ax.plot(center[0]*100+15*np.cos(np.radians(theta_arc)),
            center[1]*100+15*np.sin(np.radians(theta_arc)), 'k--', lw=1, alpha=0.3)
    for t_i, theta_i, ms in [(0, 0, '0 ms'), (0.245, 14.7, '245 ms')]:
        tp = arc_pos(theta_i)
        ax.plot(tp[0]*100, tp[1]*100, 'o', color=RED, ms=7, alpha=0.4+0.6*t_i)
        ax.annotate(ms, (tp[0]*100, tp[1]*100), fontsize=7, xytext=(6,-8), textcoords='offset points', color=RED)
    ax.plot(center[0]*100, center[1]*100, 's', color=NAVY, ms=10, zorder=5)
    cs = 0.8
    if mode == 'pursuit':
        gp = arc_pos(12); gx, gy = gp[0]*100, gp[1]*100
    else:
        gx, gy = center[0]*100, center[1]*100
    ax.plot([gx-cs, gx+cs], [gy, gy], color=color, lw=1.5)
    ax.plot([gx, gx], [gy-cs, gy+cs], color=color, lw=1.5)
    ax.set_aspect('equal'); ax.set_title(title, fontweight='bold')
    ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
    ax.set_xlim(-14, 8); ax.set_ylim(57, 82)
plt.suptitle('Figure 1: Interception task', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 4: Target Kalman filter — three conditions ──
# Shared pre-saccade phase, then fork (identical to lecture notes)
dp, dm, df = run_three_conditions(seed=42)
t_ms = np.arange(dp['n']) * dt * 1000

# LP smoother for fixation traces (removes 50ms staircase)
def lp_smooth(signal, tau_ms=5.0):
    out = np.zeros_like(signal); out[0] = signal[0]
    for i in range(1, len(signal)):
        out[i] = out[i-1] + (signal[i] - out[i-1]) * dt / (tau_ms/1000)
    return out

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# A: Position estimate
ax = axes[0, 0]
ax.plot(t_ms, dp['true_theta'], 'k--', lw=1, label='True')
ax.plot(t_ms, dp['kf_theta'], color=TEAL, lw=2, label='Pursuit (foveal + eev)')
ax.plot(t_ms, lp_smooth(dm['kf_theta']), color=PURPLE, lw=2, label='Fixation + MT vel. (10 ms)')
ax.plot(t_ms, lp_smooth(df['kf_theta']), color=ORANGE, lw=2, label='Fixation (pos. only, 50 ms)')
ax.axvline(250, color=NAVY, ls='--', lw=1, alpha=0.5)
ax.axvspan(0, 250, alpha=0.05, color=GRAY)
ax.set_ylabel('Target angle θ (°)'); ax.legend(fontsize=7)
ax.set_title('A: Position estimate (θ)', fontweight='bold'); ax.set_xlim(-20, 870)

# B: Velocity estimate
ax = axes[0, 1]
ax.plot(t_ms, dp['true_omega'], 'k--', lw=1, label='True (60°/s)')
ax.plot(t_ms, dp['kf_omega'], color=TEAL, lw=2, label='Pursuit (foveal + eev)')
ax.plot(t_ms, lp_smooth(dm['kf_omega']), color=PURPLE, lw=2, label='Fixation + MT vel. (10 ms)')
ax.plot(t_ms, lp_smooth(df['kf_omega']), color=ORANGE, lw=2, label='Fixation (pos. only, 50 ms)')
ax.axvline(250, color=NAVY, ls='--', lw=1, alpha=0.5)
ax.axvspan(0, 250, alpha=0.05, color=GRAY)
ax.set_ylabel('Target velocity ω (°/s)'); ax.legend(fontsize=7)
ax.set_title('B: Velocity estimate (ω)', fontweight='bold'); ax.set_xlim(-20, 870)

# C: Position uncertainty
ax = axes[1, 0]
ax.plot(t_ms, dp['sigma_theta'], color=TEAL, lw=2, label='Pursuit')
ax.plot(t_ms, lp_smooth(dm['sigma_theta']), color=PURPLE, lw=2, label='Fixation + MT vel. (10 ms)')
ax.plot(t_ms, lp_smooth(df['sigma_theta']), color=ORANGE, lw=2, label='Fixation (pos. only, 50 ms)')
ax.axvline(250, color=NAVY, ls='--', lw=1, alpha=0.5)
ax.axvspan(0, 250, alpha=0.05, color=GRAY)
ax.set_xlabel('Time from target onset (ms)'); ax.set_ylabel('σ_θ (°)')
ax.legend(fontsize=7); ax.set_title('C: Position uncertainty', fontweight='bold'); ax.set_xlim(-20, 870)

# D: Velocity uncertainty
ax = axes[1, 1]
ax.plot(t_ms, dp['sigma_omega'], color=TEAL, lw=2, label='Pursuit')
ax.plot(t_ms, lp_smooth(dm['sigma_omega']), color=PURPLE, lw=2, label='Fixation + MT vel. (10 ms)')
ax.plot(t_ms, lp_smooth(df['sigma_omega']), color=ORANGE, lw=2, label='Fixation (pos. only, 50 ms)')
ax.axvline(250, color=NAVY, ls='--', lw=1, alpha=0.5)
ax.axvspan(0, 250, alpha=0.05, color=GRAY)
ax.set_xlabel('Time from target onset (ms)'); ax.set_ylabel('σ_ω (°/s)')
ax.legend(fontsize=7); ax.set_title('D: Velocity uncertainty', fontweight='bold'); ax.set_xlim(-20, 870)

plt.suptitle('Figure 4: Target Kalman filter in polar coordinates — state = [θ, ω]\n'
             'Three conditions: pursuit (teal), fixation + MT velocity (purple), fixation position-only (orange)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

print(f'\nAt 250 ms (hand launch):')
print(f'  Pursuit:           σ_θ={dp["sigma_theta"][250]:.2f}°  σ_ω={dp["sigma_omega"][250]:.1f}°/s')
print(f'  Fixation + MT:     σ_θ={dm["sigma_theta"][250]:.2f}°  σ_ω={dm["sigma_omega"][250]:.1f}°/s')
print(f'  Fixation pos-only: σ_θ={df["sigma_theta"][250]:.2f}°  σ_ω={df["sigma_omega"][250]:.1f}°/s')

In [ ]:
# ══ FILL IN: what happens when MT samples at the same rate as position? ══
dm_slow = ...  # ← YOUR CODE HERE

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_ms[:dm['n']], dm['sigma_omega'][:dm['n']], color=PURPLE, lw=2, label='MT every 10 ms')
ax.plot(t_ms[:dm_slow['n']], dm_slow['sigma_omega'][:dm_slow['n']], color=PURPLE, lw=2, ls='--', label='MT every 50 ms')
ax.plot(t_ms, df['sigma_omega'], color=ORANGE, lw=2, label='Position-only (50 ms)')
ax.axvline(200, color=NAVY, ls='--', lw=1, alpha=0.3)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('σ_ω (°/s)')
ax.set_title('Direct velocity channel vs sampling rate', fontweight='bold')
ax.legend(); ax.set_xlim(0, 850); plt.tight_layout(); plt.show()
print(f'σ_ω at 200ms:  MT@10ms={dm["sigma_omega"][200]:.1f}  |  MT@50ms={dm_slow["sigma_omega"][200]:.1f}  |  pos-only={df["sigma_omega"][200]:.1f}')

---

In [ ]:
# ── Feedforward plan + Figure 5A: Launch geometry ──
plan = feedforward_plan()

# Compute aims from both conditions at hand launch (245ms)
dp_ff = run_diag('pursuit', seed=42)
dm_ff = run_diag_MT(seed=42)
t_launch_idx = 245  # index at 245ms
aim_pursuit = dp_ff['kf_theta'][t_launch_idx] + dp_ff['kf_omega'][t_launch_idx] * 0.550
aim_fixation = dm_ff['kf_theta'][t_launch_idx] + dm_ff['kf_omega'][t_launch_idx] * 0.550
theta_true = plan['true_intercept']

print(f"Pursuit:  ω̂={dp_ff['kf_omega'][t_launch_idx]:.1f}°/s  aim={aim_pursuit:.1f}°")
print(f"Fixation: ω̂={dm_ff['kf_omega'][t_launch_idx]:.1f}°/s  aim={aim_fixation:.1f}°")
print(f"True intercept: {theta_true:.1f}°  |  Difference: {aim_pursuit-aim_fixation:.1f}°")
print(f"Uncertainty: pos={plan['sig_from_pos']:.2f}° vel×T={plan['sig_from_vel']:.2f}° timing={plan['sig_from_timing']:.2f}° total={plan['sig_intercept']:.2f}°")

# Figure 5A
fig, ax = plt.subplots(figsize=(7, 7))
theta_arc = np.linspace(0, 65, 200)
ax.plot(center[0]*100+15*np.cos(np.radians(theta_arc)),
        center[1]*100+15*np.sin(np.radians(theta_arc)), 'k--', lw=1, alpha=0.3)

# Target positions
for th, label in [(0, 't=0'), (60*0.245, 't=245ms')]:
    tp = arc_pos(th); ax.plot(tp[0]*100, tp[1]*100, 'o', color=RED, ms=7)
    ax.annotate(label, (tp[0]*100, tp[1]*100), fontsize=8, xytext=(6,-8), textcoords='offset points', color=RED)

# Hand start
ax.plot(center[0]*100, center[1]*100, 's', color=NAVY, ms=10, zorder=5, label='Hand start')

# True intercept
tp_true = arc_pos(theta_true)
ax.plot(tp_true[0]*100, tp_true[1]*100, '*', color=GREEN, ms=16, zorder=5)
ax.annotate(f'True intercept ({theta_true:.0f}°)', (tp_true[0]*100, tp_true[1]*100),
           fontsize=8, xytext=(-50, 15), textcoords='offset points', color=GREEN, fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=GREEN, lw=1, alpha=0.6))

# Aim arrows
for aim, color, ls, label in [(aim_pursuit, TEAL, '-', 'Pursuit'),
                                (aim_fixation, ORANGE, '--', 'Fixation')]:
    aim_pt = arc_pos(aim)
    dx = aim_pt[0]-center[0]; dy = aim_pt[1]-center[1]; norm = np.sqrt(dx**2+dy**2)
    ae = center + R*np.array([dx,dy])/norm  # arrow to arc
    ax.annotate('', xy=(ae[0]*100, ae[1]*100), xytext=(center[0]*100, center[1]*100),
               arrowprops=dict(arrowstyle='->', color=color, lw=3, alpha=0.7, ls=ls))

# Aim labels
ax.annotate(f'Pursuit aim ({aim_pursuit:.0f}°)',
           (arc_pos(aim_pursuit)[0]*100, arc_pos(aim_pursuit)[1]*100),
           fontsize=8, xytext=(-80, -25), textcoords='offset points', color=TEAL, fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=TEAL, lw=1),
           bbox=dict(boxstyle='round,pad=0.2', facecolor='#E8F0FE', alpha=0.9))
ax.annotate(f'Fixation aim ({aim_fixation:.0f}°)',
           (arc_pos(aim_fixation)[0]*100, arc_pos(aim_fixation)[1]*100),
           fontsize=8, xytext=(10, 15), textcoords='offset points', color=ORANGE, fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFF3E0', alpha=0.9))

# Lead angle
ax.annotate(f'Lead ≈{aim_pursuit - 60*0.245:.0f}°', xy=(-5, 68), fontsize=9, color=GRAY)

ax.set_aspect('equal'); ax.set_xlim(-14, 8); ax.set_ylim(57, 82)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('Figure 5A: Launch geometry', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 5B: Shooting movement (min-jerk) ──
T_cross = plan['T_cross']; T_total = plan['T_total_mj']; A = plan['A']
t_mj = np.linspace(0, T_total, 500); x_mj = t_mj / T_total
v_mj = (A/T_total) * mj_vel(x_mj) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_mj*1000, v_mj, color=NAVY, lw=2.5, label='Velocity (cm/s)')
v_peak = max(v_mj)
ax.axvline(T_cross*1000, color=RED, ls='--', lw=1.5, alpha=0.7)
ax.axhline(0.70*v_peak, color=RED, ls=':', lw=1, alpha=0.5)
ax.plot(T_cross*1000, 0.70*v_peak, 'o', color=RED, ms=10, zorder=5)
ax.annotate(f'Arc crossing\nv = {0.70*v_peak:.0f} cm/s (70% peak)',
           (T_cross*1000, 0.70*v_peak), fontsize=9, xytext=(15,10),
           textcoords='offset points', color=RED, fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=RED))
ax.set_xlabel('Time from launch (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.set_title('Figure 5B: Shooting movement (min-jerk)', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

---

In [ ]:
# ── Figure 6: Single-trial hand trajectories ──
seed = 13
dAp = run_trial('A', 'pursuit', seed=seed); dAf = run_trial('A', 'fixation', seed=seed)
dBp = run_trial('B', 'pursuit', seed=seed); dBf = run_trial('B', 'fixation', seed=seed)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
T_launch_fb = 0.245
for row, (mlabel, dP, dF) in enumerate([
    ('Model A: Intermittent', dAp, dAf), ('Model B: Continuous', dBp, dBf)]):
    for col, (glabel, d, color) in enumerate([('Pursuit', dP, TEAL), ('Fixation', dF, ORANGE)]):
        ax = axes[row, col]
        theta_arc = np.linspace(-5, 65, 200)
        ax.plot(center[0]*100+15*np.cos(np.radians(theta_arc)),
                center[1]*100+15*np.sin(np.radians(theta_arc)), 'k--', lw=1, alpha=0.3)
        nc = d['n_cross']
        hx = np.array([center[0]+d['hand_dist'][j]*np.cos(np.radians(d['hand_true_ang'][j])) for j in range(nc)])
        hy = np.array([center[1]+d['hand_dist'][j]*np.sin(np.radians(d['hand_true_ang'][j])) for j in range(nc)])
        fx = np.array([center[0]+d['hand_dist'][j]*np.cos(np.radians(d['aim_angle'])) for j in range(nc)])
        fy = np.array([center[1]+d['hand_dist'][j]*np.sin(np.radians(d['aim_angle'])) for j in range(nc)])
        ax.plot(fx*100, fy*100, color=GRAY, lw=1.5, ls='--', alpha=0.4, label='FF only')
        ax.plot(hx*100, hy*100, color=color, lw=2.5)
        ax.plot(center[0]*100, center[1]*100, 's', color=NAVY, ms=8)
        ax.plot(hx[-1]*100, hy[-1]*100, 'o', color=color, ms=8, zorder=5)
        tp = arc_pos(60*(T_launch_fb+0.550))
        ax.plot(tp[0]*100, tp[1]*100, '*', color=GREEN, ms=14, zorder=5)
        if row==0:
            for tc,_,_,_ in d['corrections']:
                idx=int(tc/dt)
                if idx<nc:
                    ax.plot(hx[idx]*100, hy[idx]*100, 'X', color=RED, ms=10, mew=2, zorder=6)
                    ax.annotate(f'{(T_launch_fb+tc)*1000:.0f}ms', (hx[idx]*100, hy[idx]*100),
                               fontsize=7, color=RED, xytext=(5,-8), textcoords='offset points')
        hit = 'HIT' if abs(d['final_miss_deg'])<HIT_WINDOW_DEG else 'MISS'
        ax.text(0.03, 0.97, f'{abs(d["final_miss_deg"]):.1f}° ({d["final_miss_cm"]:.1f}cm) {hit}',
               transform=ax.transAxes, fontsize=9, va='top', fontweight='bold', color=color,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        ax.set_aspect('equal'); ax.set_xlim(-14,6); ax.set_ylim(57,80)
        ax.set_title(f'{glabel} — {mlabel}', fontweight='bold')
plt.suptitle('Figure 6: Hand trajectories under feedback control', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 7: Actual miss vs controller estimate ──
t_move = np.arange(dAp['n_move']) * dt * 1000
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for col, (mlabel, dP, dF) in enumerate([('Model A', dAp, dAf), ('Model B', dBp, dBf)]):
    ax = axes[col]; nc = dP['n_cross']
    ax.plot(t_move[:nc], dP['actual_miss'][:nc], color=TEAL, lw=2, label='Pursuit (actual)')
    ax.plot(t_move[:nc], dF['actual_miss'][:nc], color=ORANGE, lw=2, label='Fixation (actual)')
    ax.plot(t_move[:nc], dP['pred_miss'][:nc], color=TEAL, lw=1, ls='--', alpha=0.5, label='Pursuit (controller)')
    ax.plot(t_move[:nc], dF['pred_miss'][:nc], color=ORANGE, lw=1, ls='--', alpha=0.5, label='Fixation (controller)')
    ax.axhspan(-HIT_WINDOW_DEG, HIT_WINDOW_DEG, alpha=0.08, color=GREEN)
    ax.axhline(0, color='k', lw=0.5, alpha=0.3)
    ax.set_ylabel('Miss (°)'); ax.legend(fontsize=7)
    ax.set_title(f'Actual vs controller estimate — {mlabel}', fontweight='bold')
    ax.set_xlim(0, 550); ax.set_xlabel('Time from launch (ms)')
plt.suptitle('Figure 7: The bias the controller cannot see', fontweight='bold')
plt.tight_layout(); plt.show()

### The invisible bias
In Model A fixation, the controller estimate (dashed orange) stays within the hit window, but the actual miss is ~8°. Both the aim and the ongoing KF estimate share the Aubert-Fleischl bias — the controller thinks it's on target.

In [ ]:
# ══ FILL IN: Monte Carlo hit rates ══
N_mc = 100
for model in ['A', 'B']:
    for gaze in ['pursuit', 'fixation']:
        trials = ...  # ← YOUR CODE HERE
        hit_rate = ...  # ← YOUR CODE HERE
        mean_miss = np.mean([abs(t['final_miss_deg']) for t in trials])
        print(f'  {model}-{gaze:8s}: hit={hit_rate:.0%}, |miss|={mean_miss:.1f}°')

In [ ]:
# ── Figure 8: Miss distributions ──
N_mc = 100
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for col, model in enumerate(['A', 'B']):
    ax = axes[col]
    for gaze, color, label in [('pursuit', TEAL, 'Pursuit'), ('fixation', ORANGE, 'Fixation')]:
        misses = [run_trial(model, gaze, seed=s)['final_miss_deg'] for s in range(N_mc)]
        hit_rate = np.mean([abs(m) < HIT_WINDOW_DEG for m in misses])
        ax.hist(misses, bins=np.linspace(-15, 15, 31), alpha=0.6, color=color,
               label=f'{label} (hit={hit_rate:.0%})')
    ax.axvspan(-HIT_WINDOW_DEG, HIT_WINDOW_DEG, alpha=0.1, color=GREEN)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('Miss at crossing (°)'); ax.set_ylabel('Count')
    mlabel = 'A: Intermittent' if model == 'A' else 'B: Continuous'
    ax.set_title(f'Model {mlabel} (N={N_mc})', fontweight='bold'); ax.legend(fontsize=9)
plt.suptitle('Figure 8: Miss distributions', fontweight='bold')
plt.tight_layout(); plt.show()

---

In [ ]:
# ── Figure 9A: Bayesian prior × likelihood → posterior ──
speeds = [60, 90, 120]
sigma_lik = 12.0 / np.sqrt(14)
priors = {'Flat': {'mu': 90.0, 'sigma': 24.5}, 'Biased': {'mu': 60.0, 'sigma': 5.0}}

def bayesian_update(true_omega, prior_mu, prior_sigma, seed=42):
    rng = np.random.default_rng(seed)
    obs = [true_omega * 1.12 + rng.normal(0, 12.0) for _ in range(14)]
    obs_mean = np.mean(obs)
    prec_pr = 1/prior_sigma**2; prec_lk = 1/sigma_lik**2
    post_mu = (prec_pr*prior_mu + prec_lk*obs_mean) / (prec_pr + prec_lk)
    post_sig = 1/np.sqrt(prec_pr + prec_lk)
    return dict(post_mu=post_mu, post_sig=post_sig, obs_mean=obs_mean)

omega_range = np.linspace(20, 160, 500)
true_om = 90; biased_mean = true_om * 1.12

fig, ax = plt.subplots(figsize=(10, 5))
lik = np.exp(-0.5*((omega_range - biased_mean)/sigma_lik)**2)
ax.plot(omega_range, lik/lik.max(), color=GRAY, lw=2, ls=':', label=f'Likelihood (mean={biased_mean:.0f}°/s)')
for pname, pdict, pcolor in [('Flat', priors['Flat'], TEAL), ('Biased', priors['Biased'], ORANGE)]:
    pr = np.exp(-0.5*((omega_range-pdict['mu'])/pdict['sigma'])**2)
    ax.plot(omega_range, pr/max(pr.max(),1e-10), color=pcolor, ls='--', lw=1.5, alpha=0.5, label=f'Prior: {pname}')
    r = bayesian_update(true_om, pdict['mu'], pdict['sigma'])
    post = np.exp(-0.5*((omega_range-r['post_mu'])/r['post_sig'])**2)
    ax.fill_between(omega_range, post/post.max(), alpha=0.25, color=pcolor)
    ax.plot(omega_range, post/post.max(), color=pcolor, lw=2.5, label=f'Posterior: {r["post_mu"]:.0f}°/s')
ax.axvline(true_om, color='k', ls='--', lw=1.5, alpha=0.4, label=f'True: {true_om}°/s')
ax.set_xlabel('Velocity (°/s)'); ax.set_ylabel('Normalized density')
ax.set_title('Figure 9A: Prior × likelihood → posterior (true ω = 90°/s)', fontweight='bold')
ax.legend(fontsize=7); ax.set_xlim(30, 140); plt.tight_layout(); plt.show()

In [ ]:
# ── Figure 9B–C: Velocity estimate and feedforward miss across speeds ──
N_mc = 500; T_cross_b = 0.550; T_launch_b = 0.245
results = {}
for true_omega in speeds:
    for pname, pdict in priors.items():
        post_mus = []; misses = []
        for seed in range(N_mc):
            r = bayesian_update(true_omega, pdict['mu'], pdict['sigma'], seed=seed)
            post_mus.append(r['post_mu'])
            aim = true_omega * 0.200 + r['post_mu'] * T_cross_b
            misses.append(aim - true_omega * (T_launch_b + T_cross_b))
        results[(true_omega, pname)] = dict(
            E_omega=np.mean(post_mus), E_omega_std=np.std(post_mus),
            miss_mean=np.mean(misses), miss_std=np.std(misses))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x_pos = np.arange(len(speeds))
for pi, (pname, pcolor, marker) in enumerate([('Flat', TEAL, 'o'), ('Biased', ORANGE, 's')]):
    means = [results[(s, pname)]['E_omega'] for s in speeds]
    stds = [results[(s, pname)]['E_omega_std'] for s in speeds]
    axes[0].errorbar(x_pos + (pi-0.5)*0.15, means, yerr=stds, fmt=marker, color=pcolor, ms=8, capsize=4, lw=2, label=pname)
    means_m = [results[(s, pname)]['miss_mean'] for s in speeds]
    stds_m = [results[(s, pname)]['miss_std'] for s in speeds]
    axes[1].errorbar(x_pos + (pi-0.5)*0.15, means_m, yerr=stds_m, fmt=marker, color=pcolor, ms=8, capsize=4, lw=2, label=pname)
axes[0].plot(x_pos, speeds, 'k--', lw=1.5, alpha=0.5, label='True ω')
axes[0].plot(x_pos, [s*1.12 for s in speeds], color=RED, ls=':', lw=1.5, alpha=0.5, label='MT biased')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels([f'{s}°/s' for s in speeds])
axes[0].set_ylabel('E[ω] at launch (°/s)'); axes[0].set_title('Figure 9B: Velocity estimate', fontweight='bold'); axes[0].legend(fontsize=8)
axes[1].axhspan(-4.8, 4.8, alpha=0.06, color=GREEN); axes[1].axhline(0, color='k', lw=0.5, alpha=0.3)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f'{s}°/s' for s in speeds])
axes[1].set_ylabel('FF miss (°)'); axes[1].set_title('Figure 9C: Feedforward miss', fontweight='bold'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Van Donkelaar & Lee (1994)
Three conditions: (1) full vision + free gaze, (2) restricted hand vision, (3) restricted eye motion. Map each to our model and compare hit rates.

In [ ]:
# ══ FILL IN: Van Donkelaar & Lee conditions ══
N = 100
conditions = [
    ('Full vision + free gaze', 'B', 'pursuit'),
    ('Restricted eye motion',   'B', 'fixation'),
]
for cname, model, gaze in conditions:
    trials = ...  # ← YOUR CODE HERE
    hit_rate = ...  # ← YOUR CODE HERE
    mean_miss = np.mean([abs(t['final_miss_deg']) for t in trials])
    print(f'  {cname:30s}: hit={hit_rate:.0%}, |miss|={mean_miss:.1f}°')
print('\nRestricted hand vision (pursuit but no foveal bonus) requires modifying run_trial — left as exercise.')

---
**Key takeaways:**
1. The pursuit advantage comes from the feedback phase, not the feedforward plan
2. The Aubert-Fleischl bias is invisible to intermittent controllers during fixation
3. Bayesian priors interact with perceptual biases in counterintuitive ways
4. The foveal bonus on hand sensing is critical for the final correction